# Running QARTOD QC and pyglider Profile Wrapper 

In this notebook, QARTOD QC is first applied to the delayed-sci.nc file and saved as a new QC-enhanced NetCDF file. The QC-enhanced science NetCDF is then passed to `create_ngdac_profiles()`, which uses `pyglider` to create individual profile NetCDF files in a temporary directory. Each temporary profile is opened as an xarray Dataset, updated with ESD-specific NGDAC metadata, and written as a new NetCDF file to the final output directory. The QARTOD QC variables from the science NetCDF are retained in the final individual profile files.

In [1]:
from pathlib import Path
from esdglider.utils import (
    update_ngdac_profile_attributes,
    create_ngdac_profiles,
)
from esdglider.qartod import run_qartod_qc

import pyglider.ncprocess as pgncprocess
import netCDF4
import numpy as np
import xarray as xr
import yaml

## Run QARTOD QC on delayed-sci.nc file

In [2]:
run_qartod_qc(
    input_file="/Users/madisonrichardson/Desktop/esdglider/sam-nc-files/amlr08-20220513/processed-L1/amlr08-20220513-delayed-sci.nc",
    output_file="/Users/madisonrichardson/Desktop/esdglider/sam-nc-files/amlr08-20220513/processed-L1/qc/amlr08-20220513-delayed-sci-qc.nc",
    config_file="/Users/madisonrichardson/Desktop/esdglider-madi/esdglider/data/qartod-config.yml"
)

Variable 'potential_temperature' is not defined in the QARTOD configuration. Using valid_min/valid_max from 'temperature' to build default gross range thresholds.
Variable 'potential_density' is not defined in the QARTOD configuration. Using valid_min/valid_max from 'density' to build default gross range thresholds.
Variable 'chlorophyll' is not defined in the QARTOD configuration and does not contain usable valid_min/valid_max attributes. Using placeholder thresholds.
Variable 'cdom' is not defined in the QARTOD configuration and does not contain usable valid_min/valid_max attributes. Using placeholder thresholds.
Variable 'backscatter_700' is not defined in the QARTOD configuration and does not contain usable valid_min/valid_max attributes. Using placeholder thresholds.
Variable 'oxygen_concentration' is not defined in the QARTOD configuration and does not contain usable valid_min/valid_max attributes. Using placeholder thresholds.
Variable 'oxygen_saturation' is not defined in the Q

## Define input and output files

We use the delayed-sci-qc.nc file for splitting into individual profile files. 

**Disclaimer**: `pyglider.ncprocess.extract_timeseries_profiles()` expects **instrument_ctd** metadata to be defined under **profile_variables** in the deployment YAML. ESD deployment YAML files currently store instrument metadata under **glider_devices**, so **instrument_ctd** may need to be duplicated under **profile_variables** for the pyglider function to run successfully. This is currently a compatibility workaround; a future implementation could handle this within the esdglider wrapper so the deployment YAML does not need to be modified.


In [3]:
# Input deployment timeseries
input_file = Path(
    "/Users/madisonrichardson/Desktop/esdglider/sam-nc-files/amlr08-20220513/processed-L1/qc/amlr08-20220513-delayed-sci-qc.nc"
)

# Deployment YAML
deployment_yaml = Path(
    "/Users/madisonrichardson/Desktop/esdglider/sam-nc-files/amlr08-20220513/amlr08-20220513.yml"
)

# Use a separate directory while testing
output_dir = Path(
    "/Users/madisonrichardson/Desktop/esdglider/sam-nc-files/amlr08-20220513/ngdac-profile-testing"
)
output_dir.mkdir(
    parents=True,
    exist_ok=True,
)

## Run wrapper function `create_ngdac_profiles`

In [4]:
create_ngdac_profiles(
    inname=input_file,
    outdir=output_dir,
    deploymentyaml=deployment_yaml,
    force=True,
)

## Verify the generated profiles

In [5]:
profile_files = sorted(
    output_dir.glob("*.nc")
)

print(
    f"Created {len(profile_files)} profile files"
)

for profile_file in profile_files[:5]:
    print(profile_file.name)

Created 102 profile files
amlr08-20220513T1858.nc
amlr08-20220513T1902.nc
amlr08-20220513T1918.nc
amlr08-20220513T1925.nc
amlr08-20220513T1946.nc


### Inspect profile

In [6]:
profile_file = profile_files[0]

ds = xr.open_dataset(
    profile_file,
)

ds

<xarray.Dataset> Size: 29kB
Dimensions:                      (time: 104)
Coordinates:
  * time                         (time) datetime64[ns] 832B 2022-05-13T18:58:...
Data variables: (12/61)
    latitude                     (time) float64 832B ...
    longitude                    (time) float64 832B ...
    depth                        (time) float64 832B ...
    profile_index                (time) float64 832B ...
    conductivity                 (time) float64 832B ...
    temperature                  (time) float64 832B ...
    ...                           ...
    lon_qc                       (time) int8 104B ...
    lat_qc                       (time) int8 104B ...
    instrument_flbbcd            int32 4B ...
    instrument_oxygen            int32 4B ...
    instrument_shadowgraph       int32 4B ...
    instrument_echosounder       int32 4B ...
Attributes: (12/57)
    Conventions:               CF-1.8
    Metadata_Conventions:      Unidata Dataset Discovery v1.0, COARDS, CF-1.8
    acknowledgment:            This work was supported by funding from NOAA.
    cdm_data_type:             Trajectory
    comment:                    
    contributor_name:          Christian Reiss, George Watters, Jennifer Wals...
    ...                        ...
    time_coverage_end:         2022-05-16T18:26:17.496000000
    time_coverage_start:       2022-05-13T18:57:22.000000000
    title:                     amlr08-20220513T1857
    transmission_system:       IRIDIUM
    wmo_id:                     
    ioos_qc_version:           2.3.0

In [7]:
print(ds["profile_id"].values)
print(np.unique(ds["profile_index"].values))

1.0
[1.]


### Check attributes

In [8]:
ds["platform"].attrs

{'comment': 'Teledyne Webb Research Slocum G3 glider operated by NOAA SWFSC Ecosystem Science Division',
 'id': 'amlr08',
 'instrument': 'instrument_ctd, instrument_flbbcd, instrument_oxygen, instrument_shadowgraph, instrument_echosounder',
 'long_name': 'Teledyne Webb Research Slocum G3 glider amlr08',
 'type': 'platform',
 'wmo_id': ' '}

In [9]:
print("id:")
print(ds["platform"].attrs["id"])

print("\ninstrument:")
print(ds["platform"].attrs["instrument"])

print("\nlong_name:")
print(ds["platform"].attrs["long_name"])

# Should be empty
print("\nInstrument global attributes:")
print(
    [
        attr
        for attr in ds.attrs
        if attr.startswith("instrument_")
    ]
)

id:
amlr08

instrument:
instrument_ctd, instrument_flbbcd, instrument_oxygen, instrument_shadowgraph, instrument_echosounder

long_name:
Teledyne Webb Research Slocum G3 glider amlr08

Instrument global attributes:
[]


### Check out instruments

In [10]:
instrument_variables = [
    var
    for var in ds.data_vars
    if var.startswith("instrument_")
]

instrument_variables

['instrument_ctd',
 'instrument_flbbcd',
 'instrument_oxygen',
 'instrument_shadowgraph',
 'instrument_echosounder']

### Check profile name

In [11]:
ds["trajectory"]
ds["trajectory"].attrs

with xr.open_dataset(input_file) as science_ds:
    print("Science ID:")
    print(science_ds.attrs["id"])

print("\nProfile trajectory:")
print(ds["trajectory"].values)

Science ID:
amlr08-20220513T1857

Profile trajectory:
np.bytes_(b'amlr08-20220513T1857')


## Check QARTOD variables exist

In [12]:
qc_variables = [
    var
    for var in ds.data_vars
    if var.endswith("_qc")
]

qc_variables

['latitude_qc',
 'longitude_qc',
 'pressure_qc',
 'depth_qc',
 'temperature_qc',
 'conductivity_qc',
 'salinity_qc',
 'density_qc',
 'potential_temperature_qc',
 'potential_density_qc',
 'chlorophyll_qc',
 'cdom_qc',
 'backscatter_700_qc',
 'oxygen_concentration_qc',
 'oxygen_saturation_qc',
 'water_velocity_eastward_qc',
 'water_velocity_northward_qc',
 'time_qc',
 'lon_qc',
 'lat_qc']

In [13]:
ds["temperature_qc"]

<xarray.DataArray 'temperature_qc' (time: 104)> Size: 416B
[104 values with dtype=float32]
Coordinates:
  * time     (time) datetime64[ns] 832B 2022-05-13T18:58:40 ... 2022-05-13T19...
Attributes:
    long_name:           QARTOD aggregate quality flag for temperature
    standard_name:       sea_water_temperature status_flag
    flag_values:         [1 2 3 4 9]
    flag_meanings:       GOOD UNKNOWN SUSPECT FAIL MISSING
    valid_min:           1
    valid_max:           9
    comment:             Aggregate QARTOD flag generated using ioos_qc package.
    flag_configuration:  {"configuration_source":"template_modified_in_memory...
    average_method:      QC_protocol

In [14]:
ds["potential_temperature_qc"]

<xarray.DataArray 'potential_temperature_qc' (time: 104)> Size: 416B
[104 values with dtype=float32]
Coordinates:
  * time     (time) datetime64[ns] 832B 2022-05-13T18:58:40 ... 2022-05-13T19...
Attributes:
    long_name:           QARTOD aggregate quality flag for potential_temperature
    standard_name:       sea_water_potential_temperature status_flag
    flag_values:         [1 2 3 4 9]
    flag_meanings:       GOOD UNKNOWN SUSPECT FAIL MISSING
    valid_min:           1
    valid_max:           9
    comment:             Aggregate QARTOD flag generated using ioos_qc package.
    flag_configuration:  {"configuration_source":"template_modified_in_memory...
    average_method:      QC_protocol

In [15]:
ds["potential_density_qc"]

<xarray.DataArray 'potential_density_qc' (time: 104)> Size: 416B
[104 values with dtype=float32]
Coordinates:
  * time     (time) datetime64[ns] 832B 2022-05-13T18:58:40 ... 2022-05-13T19...
Attributes:
    long_name:           QARTOD aggregate quality flag for potential_density
    standard_name:       sea_water_potential_density status_flag
    flag_values:         [1 2 3 4 9]
    flag_meanings:       GOOD UNKNOWN SUSPECT FAIL MISSING
    valid_min:           1
    valid_max:           9
    comment:             Aggregate QARTOD flag generated using ioos_qc package.
    flag_configuration:  {"configuration_source":"template_modified_in_memory...
    average_method:      QC_protocol

In [16]:
ds["density_qc"]

<xarray.DataArray 'density_qc' (time: 104)> Size: 416B
[104 values with dtype=float32]
Coordinates:
  * time     (time) datetime64[ns] 832B 2022-05-13T18:58:40 ... 2022-05-13T19...
Attributes:
    long_name:           QARTOD aggregate quality flag for density
    standard_name:       sea_water_density status_flag
    flag_values:         [1 2 3 4 9]
    flag_meanings:       GOOD UNKNOWN SUSPECT FAIL MISSING
    valid_min:           1
    valid_max:           9
    comment:             Aggregate QARTOD flag generated using ioos_qc package.
    flag_configuration:  {"configuration_source":"template_modified_in_memory...
    average_method:      QC_protocol

In [17]:
ds["conductivity_qc"]

<xarray.DataArray 'conductivity_qc' (time: 104)> Size: 416B
[104 values with dtype=float32]
Coordinates:
  * time     (time) datetime64[ns] 832B 2022-05-13T18:58:40 ... 2022-05-13T19...
Attributes:
    long_name:           QARTOD aggregate quality flag for conductivity
    standard_name:       sea_water_electrical_conductivity status_flag
    flag_values:         [1 2 3 4 9]
    flag_meanings:       GOOD UNKNOWN SUSPECT FAIL MISSING
    valid_min:           1
    valid_max:           9
    comment:             Aggregate QARTOD flag generated using ioos_qc package.
    flag_configuration:  {"configuration_source":"template_modified_in_memory...
    average_method:      QC_protocol

## Verify that QC flags were retained

In [18]:
for var in qc_variables:
    flags, counts = np.unique(
        ds[var].values,
        return_counts=True,
    )

    print(
        var,
        dict(zip(flags, counts)),
    )

latitude_qc {np.float32(1.0): np.int64(104)}
longitude_qc {np.float32(1.0): np.int64(104)}
pressure_qc {np.float32(1.0): np.int64(95), np.float32(3.0): np.int64(9)}
depth_qc {np.float32(1.0): np.int64(95), np.float32(3.0): np.int64(9)}
temperature_qc {np.float32(1.0): np.int64(104)}
conductivity_qc {np.float32(1.0): np.int64(103), np.float32(3.0): np.int64(1)}
salinity_qc {np.float32(1.0): np.int64(104)}
density_qc {np.float32(1.0): np.int64(104)}
potential_temperature_qc {np.float32(1.0): np.int64(103), np.float32(3.0): np.int64(1)}
potential_density_qc {np.float32(1.0): np.int64(104)}
chlorophyll_qc {np.float32(1.0): np.int64(104)}
cdom_qc {np.float32(1.0): np.int64(95), np.float32(3.0): np.int64(9)}
backscatter_700_qc {np.float32(1.0): np.int64(98), np.float32(3.0): np.int64(5), np.float32(4.0): np.int64(1)}
oxygen_concentration_qc {np.float32(1.0): np.int64(103), np.float32(3.0): np.int64(1)}
oxygen_saturation_qc {np.float32(1.0): np.int64(103), np.float32(3.0): np.int64(1)}
water_